#### 1. 定义状态和上下文

In [1]:
from dataclasses import dataclass
from pydantic import BaseModel, Field
from langchain.agents import AgentState


@dataclass
class Context:
    """上下文用于传递用户 ID 等跨节点信息"""

    user_id: str


class MessagesState(AgentState):
    """对话状态，包含消息历史"""

    pass


class MemoryExtraction(BaseModel):
    """Result of memory extraction from user message."""
    should_save: bool = Field(description="Whether the message contains information worth remembering")
    memory: str = Field(description="The memory to save, empty string if should_save is false")

#### 2. 初始化 store、checkpointer 和 LLM

In [2]:
from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from agent_lab.model_hub import LLM_FAST, EMBEDDING_SMALL


store = InMemoryStore(
    index={
        'embed': init_embeddings(**EMBEDDING_SMALL),
        'dims': 1536,
        'fields': ['content', '$'],
    }
)

# Checkpointer 用于保存线程状态（对话历史）
checkpointer = InMemorySaver()

# 初始化聊天模型
llm = init_chat_model(**LLM_FAST, temperature=0)
model_with_structure = llm.with_structured_output(MemoryExtraction)

#### 3. 定义图节点

In [3]:
import uuid
from langgraph.runtime import Runtime
from langchain.messages import HumanMessage, SystemMessage


async def update_memory(state: MessagesState, runtime: Runtime[Context]):
    """更新用户记忆节点 - 用 LLM 智能判断是否需要保存记忆"""
    user_id = runtime.context.user_id
    namespace = (user_id, 'memories')

    last_message = state['messages'][-1]

    # 只有用户消息才可能包含新信息
    if isinstance(last_message, HumanMessage):
        user_message = last_message.content  # type: ignore

        # 用 LLM 判断是否需要保存记忆
        system_prompt = """You are a memory extraction assistant. Your job is to determine if the user's message contains information worth remembering about themselves.

Extract information such as:
- Name, identity, personal facts
- Preferences, likes, dislikes
- Location, living situation
- Important life events or plans

Example 1:
Input: "Hi! How are you?"
Output: should_save=false, memory=""

Example 2:
Input: "My name is Leo and I like pizza!"
Output: should_save=true, memory="Leo likes pizza"

Example 3:
Input: "I want to eat my favorite food!"
Output: should_save=false, memory="" """

        # 使用结构化输出
        result = await model_with_structure.ainvoke([  # type: ignore
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_message)
        ])

        if result.should_save and result.memory:
            memory_id = str(uuid.uuid4())
            memory = {'content': result.memory}
            await runtime.store.aput(namespace, memory_id, memory)  # type: ignore
            print(f'[Memory Saved] {memory}')


async def call_model(state: MessagesState, runtime: Runtime[Context]):
    """调用模型节点 - 从 store 检索记忆并生成回复"""
    user_id = runtime.context.user_id
    namespace = (user_id, 'memories')

    # 从 store 搜索相关记忆
    memories = await runtime.store.asearch(namespace, query=state['messages'][-1].content, limit=3) # type: ignore

    # 相似度阈值过滤，只保留足够相关的记忆
    SIMILARITY_THRESHOLD = 0.2
    relevant_memories = [m for m in memories if m.score is None or m.score > SIMILARITY_THRESHOLD]

    # 构建记忆上下文
    memory_context = ''
    if relevant_memories:
        memory_context = '\n'.join([f'- {d.value["content"]}' for d in relevant_memories])
        print(f'[Memory Retrieved] Found {len(relevant_memories)} relevant memories (threshold: {SIMILARITY_THRESHOLD}):')
        for m in relevant_memories:
            print(f'  - score={m.score:.3f}: {m.value["content"]}')
    else:
        print(f'[Memory Retrieved] No relevant memories found (threshold: {SIMILARITY_THRESHOLD})')

    # 构建系统提示词
    system_prompt = """You are a friendly assistant who remembers things about the user.
If you have memories about the user, use them naturally in conversation.
When the user shares information about themselves (name, preferences, location, etc.),
you should acknowledge that you'll remember it."""

    if memory_context:
        system_prompt += f"\n\nHere's what you remember about the user:\n{memory_context}"

    # 调用 LLM
    messages = [SystemMessage(content=system_prompt)] + list(state['messages'])  # type: ignore
    response = await llm.ainvoke(messages)  # type: ignore

    return {'messages': [response]}


#### 4. 构建并编译图

In [4]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(MessagesState, context_schema=Context)

# 添加节点
builder.add_node('update_memory', update_memory)
builder.add_node('call_model', call_model)

# 添加边
builder.add_edge(START, 'update_memory')
builder.add_edge('update_memory', 'call_model')
builder.add_edge('call_model', END)

# 编译图，同时传入 checkpointer 和 store
graph = builder.compile(checkpointer=checkpointer, store=store)

#### 5. 运行示例

In [5]:
async def chat(user_input: str, config: dict, user_id: str) -> None:
    """辅助函数：发送消息并打印回复"""
    print(f'\n[User {user_id}] {user_input}')
    async for chunk in graph.astream(
        {'messages': [HumanMessage(content=user_input)]},
        config, # type: ignore
        stream_mode='updates',
        version='v2',
        context=Context(user_id=user_id),
    ):
        if chunk['type'] == 'updates':
            for _step, data in chunk['data'].items():
                if data and 'messages' in data:
                    print(f'[AI] {data["messages"][-1].content}')

In [6]:
print('\n--- Thread 1: Conversation 1 ---')
config1 = {'configurable': {'thread_id': 'thread-1'}}

await chat('Hi!', config1, 'user-1')
await chat('My name is Leo and I like pizza!', config1, 'user-1')


--- Thread 1: Conversation 1 ---

[User user-1] Hi!
[Memory Retrieved] No relevant memories found (threshold: 0.2)
[AI] Hey there! Great to meet you. How are you doing today?

[User user-1] My name is Leo and I like pizza!
[Memory Saved] {'content': 'Leo likes pizza'}
[Memory Retrieved] Found 1 relevant memories (threshold: 0.2):
  - score=0.770: Leo likes pizza
[AI] Nice to meet you, Leo! I'll remember that you're a pizza fan. What's your favorite kind of pizza?


In [7]:
print('\n--- Thread 2: Conversation 2 (same user, new conversation) ---')
config2 = {'configurable': {'thread_id': 'thread-2'}}

await chat('Do you know me? I want to eat my favorite food!', config2, 'user-1')


--- Thread 2: Conversation 2 (same user, new conversation) ---

[User user-1] Do you know me? I want to eat my favorite food!
[Memory Retrieved] Found 1 relevant memories (threshold: 0.2):
  - score=0.345: Leo likes pizza
[AI] Of course I remember you, Leo! You're a big fan of pizza, right? 🍕 Sounds like it's time to grab a slice (or a whole pie) of your favorite food! What kind of pizza are you in the mood for today?


In [8]:
print('\n--- Thread 3: Conversation 3 (different user) ---')
config3 = {'configurable': {'thread_id': 'thread-3'}}

await chat('Do you know me?', config3, 'user-2')


--- Thread 3: Conversation 3 (different user) ---

[User user-2] Do you know me?
[Memory Retrieved] No relevant memories found (threshold: 0.2)
[AI] I don't know you yet, but I'd love to get to know you! Feel free to tell me your name, where you're from, or anything else you'd like to share. I'll remember what you tell me so I can make our conversations more personal and helpful. 😊
